### Save in Two Formats

In [ ]:
# !pip install numpy
# !pip install pandas
# !pip install pyarrow
# !pip install python-dotenv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.9/35.9 MB 90.7 MB/s  0:00:00m0:00:0100:01


In [1]:
import os, pathlib, datetime as dt
import pandas as pd
from dotenv import load_dotenv

In [2]:
load_dotenv()
RAW_DIR = pathlib.Path(os.getenv("DATA_DIR_RAW", "data/raw"))
PROC_DIR = pathlib.Path(os.getenv("DATA_DIR_PROCESSED", "data/processed"))
RAW_DIR.mkdir(parents=True, exist_ok=True)
PROC_DIR.mkdir(parents=True, exist_ok=True)
print("RAW_DIR:", RAW_DIR.resolve())
print("PROC_DIR:", PROC_DIR.resolve())

RAW_DIR: /Users/jenyunting/Desktop/bootcamp_yunting_jen/homework/homework5/data/raw
PROC_DIR: /Users/jenyunting/Desktop/bootcamp_yunting_jen/homework/homework5/data/processed


In [3]:
import numpy as np

# Seed the generator so this notebook produces the SAME numbers on every run -
# on the projector, and on your machine at home. Without this line the prices
# change each time you run the cell, and so does every file you save from it.
np.random.seed(5)

dates = pd.date_range("2024-01-01", periods=10, freq="D")
df = pd.DataFrame({
    'date': dates,
    'ticker': ['AAPL']*10,
    'price': 150 + np.random.randn(10).cumsum()
})
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   date    10 non-null     datetime64[us]
 1   ticker  10 non-null     str           
 2   price   10 non-null     float64       
dtypes: datetime64[us](1), float64(1), str(1)
memory usage: 412.0 bytes


In [4]:
def ts(): return dt.datetime.now().strftime('%Y%m%d-%H%M%S')

try:
    csv_path = RAW_DIR / f"sample_{ts()}.csv"
    df.to_csv(csv_path, index=False)
    print(f"Naively saved data as CSV → {csv_path}")
except Exception as e:
    print("Failed to save data as CSV:", e)

# Try saving to Parquet
try:
    parq_path = PROC_DIR / f"sample_{ts()}.parquet"
    df.to_parquet(parq_path)
    print(f"Naively saved data as Parquet → {parq_path}")
except Exception as e:
    print("Failed to save data as Parquet:", e)

Naively saved data as CSV → ../data/raw/sample_20260817-103650.csv
Naively saved data as Parquet → ../data/processed/sample_20260817-103650.parquet


### Reload and Validate

In [5]:
def validate_loaded(original: pd.DataFrame, reloaded: pd.DataFrame, cols=('date','ticker','price')):
    checks = {
        'shape_equal': original.shape == reloaded.shape,
        'cols_present': all(c in reloaded.columns for c in cols)
    }
    # dtype sanity checks
    if 'price' in reloaded.columns:
        checks['price_is_numeric'] = pd.api.types.is_numeric_dtype(reloaded['price'])
    if 'date' in reloaded.columns:
        checks['date_is_datetime'] = pd.api.types.is_datetime64_any_dtype(reloaded['date'])
    return checks

df_csv = pd.read_csv(csv_path, parse_dates=['date'])
print('CSV validation:', validate_loaded(df, df_csv))

CSV validation: {'shape_equal': True, 'cols_present': True, 'price_is_numeric': True, 'date_is_datetime': True}


In [8]:
if parq_path:
    try:
        df_pq = pd.read_parquet(parq_path)
        print('Parquet validation:', validate_loaded(df, df_pq))
    except Exception as e:
        print('Parquet read failed:', e)

Parquet validation: {'shape_equal': True, 'cols_present': True, 'price_is_numeric': True, 'date_is_datetime': True}


### Refactor to Utilities

In [9]:
from typing import Union

def ensure_dir(path: pathlib.Path):
    path.parent.mkdir(parents=True, exist_ok=True)

def detect_format(path: Union[str, pathlib.Path]):
    suf = str(path).lower()
    if suf.endswith('.csv'): return 'csv'
    if suf.endswith('.parquet') or suf.endswith('.pq') or suf.endswith('.parq'): return 'parquet'
    raise ValueError('Unsupported format for: ' + str(path))

def write_df(df: pd.DataFrame, path: Union[str, pathlib.Path]):
    path = pathlib.Path(path)
    ensure_dir(path)
    fmt = detect_format(path)
    if fmt == 'csv':
        df.to_csv(path, index=False)
    elif fmt == 'parquet':
        try:
            df.to_parquet(path)
        except Exception as e:
            raise RuntimeError('Parquet engine not available. Install pyarrow or fastparquet.') from e
    return path

def read_df(path: Union[str, pathlib.Path]):
    path = pathlib.Path(path)
    fmt = detect_format(path)
    if fmt == 'csv':
        return pd.read_csv(path, parse_dates=['date']) if 'date' in pd.read_csv(path, nrows=0).columns else pd.read_csv(path)
    elif fmt == 'parquet':
        try:
            return pd.read_parquet(path)
        except Exception as e:
            raise RuntimeError('Parquet engine not available. Install pyarrow or fastparquet.') from e

# Demo utility usage
csv2 = RAW_DIR / f"prices_util_{ts()}.csv"
pq2  = PROC_DIR / f"prices_util_{ts()}.parquet"
write_df(df, csv2)
df2 = read_df(csv2)
print('Reloaded CSV via util, shape:', df2.shape)

try:
    write_df(df, pq2)
    df3 = read_df(pq2)
    print('Reloaded Parquet via util, shape:', df3.shape)
except RuntimeError as e:
    print('Parquet util demo skipped:', e)

Reloaded CSV via util, shape: (10, 3)
Reloaded Parquet via util, shape: (10, 3)
